# Pipeline

Main pipeline notebook. All logic lives in `pipeline/`; this notebook handles configuration and orchestration only.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%reload_ext autoreload

In [3]:
import torch
import pandas as pd
import os
import sys

from pathlib import Path

# Moving up to the project root to ensure imports work correctly regardless of execution context
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


from pathlib import Path
from openai import OpenAI
from utilities import (
    OPENROUTER_API_KEY,
    OPENROUTER_BASE_URL,
    TASK_STATEMENTS_PATH,
    MAJOR_CATEGORIES,
    WORK_RELATED_LABELED_OUTPUT_PATH,
    TIMEZONES_LABELED_OUTPUT_PATH,
    TASK_MAPPING_LABELED_OUTPUT_PATH,
    LABOR_TRANSFER_LABELED_OUTPUT_PATH,
    JOB_ZONES_PATH,
    ExecutionMode,
    FINAL_LABELED_OUTPUT_PATH,
)
from pipeline import (
    load_wildchat,
    sample_conversations,
    preprocess_conversations,
    filter_work_conversations,
    find_timezones,
    normalize_timezone,
    map_conversation_to_task,
    filter_task_mappings,
    analyze_labor_transfer,
    expand_labor_transfer_labels,
)
from validation import (
    work_related_metrics,
    agreement_rate,
    print_metrics,
)

In [4]:
# Device setup
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [5]:
# API client setup
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

In [6]:
execution_mode = ExecutionMode.DIRECT

## 2. Data Loading

In [7]:
# TODO: When we decide on the validation sample, swap out the
# sample_conversations_df with that

In [8]:
english_conversations = load_wildchat()
total_rows = len(english_conversations)
print(f"Total English conversations: {total_rows}")

Loading dataset from disk:   0%|          | 0/46 [00:00<?, ?it/s]

Total English conversations: 1679371


In [9]:
sample_df = sample_conversations(english_conversations)
print(f"Sample shape: {sample_df.shape}")

Sample shape: (58777, 14)


In [10]:
sample_conversations_df = preprocess_conversations(sample_df)
print(f"After dedup: {sample_conversations_df.shape}")
sample_conversations_df.head(2)

After dedup: (52489, 5)


,conversation,timestamp,country,state,hashed_ip
0,"[{'role': 'user', 'content': 'can you name any...",2025-03-12 09:33:27,United Kingdom,Royal Kensington and Chelsea,62d519d4f9104d3ad08f25664bf790a39805b5d1e39c21...
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...


In [11]:
sample_conversations_df = sample_conversations_df[:50]
sample_conversations_df.shape

(50, 5)

## 3. Work-Related Conversation Filtering

In [12]:
if not WORK_RELATED_LABELED_OUTPUT_PATH.exists():
    answers = filter_work_conversations(
        client=client,
        conversations=sample_conversations_df,
        path=WORK_RELATED_LABELED_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
    answers_df = pd.DataFrame(
        {
            "conversation": sample_conversations_df["conversation"],
            "is_work_related_model": answers,
        }
    )
    answers_df.to_csv(WORK_RELATED_LABELED_OUTPUT_PATH, index=False)
else:
    answers_df = pd.read_csv(WORK_RELATED_LABELED_OUTPUT_PATH)

sample_conversations_df["is_work_related_model"] = answers_df[
    "is_work_related_model"
].values

work_related_df = sample_conversations_df[
    sample_conversations_df["is_work_related_model"] == "Yes"
].copy()

print(f"Work-related conversations: {work_related_df.shape}")
work_related_df.head(2)

Running 50 direct Work Related calls...


Work Related:  76%|███████▌  | 38/50 [04:19<02:06, 10.52s/it]

Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 2). Retrying in 2.00 seconds...
Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 2). Retrying in 2.00 seconds...
Rate limit hit (attempt 2). Retrying in 2.00 seconds...
Rate limit hit (attempt 3). Retrying in 4.00 seconds...
Rate limit hit (attempt 3). Retrying in 4.00 seconds...
Rate limit hit (attempt 3). Retrying in 4.00 seconds...
Rate limit hit (attempt 4). Retrying in 8.00 seconds...
Rate limit hit (attempt 4). Retrying in 8.00 seconds...


Work Related: 100%|██████████| 50/50 [05:26<00:00,  6.53s/it]

Work-related conversations: (16, 6)


,conversation,timestamp,country,state,hashed_ip,is_work_related_model
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...,Yes
5,"[{'role': 'user', 'content': 'System: You are ...",2024-11-04 01:14:03,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes


In [13]:
sample_conversations_df["is_work_related_model"].value_counts()

is_work_related_model
No       23
Yes      16
Maybe    11
Name: count, dtype: int64

In [ ]:
metrics = work_related_metrics(
    y_true=sample_conversations_df["is_work_related_human"],
    y_pred=sample_conversations_df["is_work_related_model"],
)
print_metrics(metrics)

## 4. Timezone conversion

In [15]:
if not TIMEZONES_LABELED_OUTPUT_PATH.exists():
    work_related_df = find_timezones(df=work_related_df)
    work_related_df.to_csv(TIMEZONES_LABELED_OUTPUT_PATH, index=False)
else:
    work_related_df = pd.read_csv(TIMEZONES_LABELED_OUTPUT_PATH)

work_related_df = work_related_df[work_related_df["timezone"].notnull()]
print(f"After timezone filter: {work_related_df.shape}")

Geocoding query: 'Khyber Pakhtunkhwa, Pakistan' -> Location: خیبر پختونخوا, پاکستان
Found location for query 'Khyber Pakhtunkhwa, Pakistan': خیبر پختونخوا, پاکستان (lat: 33.712802, lng: 71.2678805)
Geocoding query: 'nan' -> Location: ننگرهار ولايت, افغانستان
Found location for query 'nan': ننگرهار ولايت, افغانستان (lat: 34.220389, lng: 70.3800314)
Geocoding query: 'Capital Region, Denmark' -> Location: Sendiráð Danmerkur, 29, Hverfisgata, Austurbær, Miðborg, Reykjavíkurborg, Höfuðborgarsvæðið, 101, Ísland
Found location for query 'Capital Region, Denmark': Sendiráð Danmerkur, 29, Hverfisgata, Austurbær, Miðborg, Reykjavíkurborg, Höfuðborgarsvæðið, 101, Ísland (lat: 64.146742, lng: -21.92952)
Geocoding query: 'Brussels Capital, Belgium' -> Location: Région de Bruxelles-Capitale - Brussels Hoofdstedelijk Gewest, België / Belgique / Belgien
Found location for query 'Brussels Capital, Belgium': Région de Bruxelles-Capitale - Brussels Hoofdstedelijk Gewest, België / Belgique / Belgien (lat:

In [16]:
work_related_df = normalize_timezone(df=work_related_df)
work_related_df[["timestamp", "timezone", "timestamp_local"]].head(3)

,timestamp,timezone,timestamp_local
0,2024-10-01 15:01:08+00:00,Asia/Karachi,2024-10-01 20:01:08+05:00
1,2024-11-04 01:14:03+00:00,Asia/Kabul,2024-11-04 05:44:03+04:30
2,2024-11-05 23:54:06+00:00,Atlantic/Reykjavik,2024-11-05 23:54:06+00:00


## 5. Task Mapping

In [17]:
tasks_df = pd.read_csv(TASK_STATEMENTS_PATH)
tasks_df.drop(
    columns=["Incumbents Responding", "Date", "Domain Source", "Task Type"],
    inplace=True,
)
tasks_df["major_category"] = tasks_df["O*NET-SOC Code"].apply(
    lambda x: MAJOR_CATEGORIES[x[0:2]]
)
tasks_df.head()

,O*NET-SOC Code,Title,Task ID,Task,major_category
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Management Occupations
1,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assig...,Management Occupations
2,11-1011.00,Chief Executives,8825,Analyze operations to evaluate performance of ...,Management Occupations
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Management Occupations
4,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Management Occupations


In [18]:
if not TASK_MAPPING_LABELED_OUTPUT_PATH.exists():
    task_mapped_df = map_conversation_to_task(
        client=client,
        conversations=work_related_df,
        tasks=tasks_df,
        path=TASK_MAPPING_LABELED_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
else:
    task_mapped_df = pd.read_csv(TASK_MAPPING_LABELED_OUTPUT_PATH)

print(f"Task mapped conversations: {task_mapped_df.shape}")
task_mapped_df.head(2)

Running 15 direct Profession mapping calls via OpenRouter...


Profession Mapping:  13%|█▎        | 2/15 [01:08<07:45, 35.77s/it]

Profession Mapping:  73%|███████▎  | 11/15 [06:47<01:53, 28.41s/it]

Profession Mapping: 100%|██████████| 15/15 [12:00<00:00, 48.01s/it]


Running 15 direct Task mapping calls via OpenRouter...


Task Mapping: 100%|██████████| 15/15 [08:28<00:00, 33.90s/it]

Task mapped conversations: (15, 3)


,conversation,professions,tasks
0,"[{'role': 'user', 'content': 'sir followings a...","[Editors, Technical Writers, Copy Writers, Pro...",[Proofreaders and Copy Markers: Correct or rec...
1,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Public Relation...",[Interpreters and Translators:Read written mat...


In [19]:
task_mapped_df = filter_task_mappings(df=task_mapped_df, column_name="tasks")

task_mapped_df["job_title"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[0] if pd.notnull(x) else None
)
task_mapped_df["selected_task"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[1] if pd.notnull(x) else None
)

print(f"After consensus filter: {task_mapped_df.shape}")
task_mapped_df.head(2)

Error processing items: ['Generating Midjourney prompts', 'Generating Midjourney prompts', 'Generating Midjourney prompts', 'Generating Midjourney prompts', 'Generating Midjourney prompts']
Error processing items: ['Identify and analyze areas of potential risk to the assets, earning capacity, or success of organizations.', 'Devise scenario analyses reflecting possible severe market events.', 'Develop or implement risk-assessment models or methodologies.', 'Produce reports or presentations that outline findings, explain risk positions, or recommend changes.', 'Plan, and contribute to development of, risk management systems.']
After consensus filter: (9, 5)


,conversation,professions,tasks,job_title,selected_task
1,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Public Relation...",Interpreters and Translators:Read written mate...,Interpreters and Translators,"Read written materials, such as legal document..."
3,"[{'role': 'user', 'content': 'You are a helpfu...","[Instructional Designers and Technologists, Te...",Instructional Designers and Technologists:Deve...,Instructional Designers and Technologists,Develop instructional materials and products f...


In [20]:
task_mapped_df["conversation_str"] = task_mapped_df["conversation"].astype(str)
work_related_df["conversation_str"] = work_related_df["conversation"].astype(str)

task_mapped_df = task_mapped_df.merge(
    work_related_df.drop(columns=["conversation"]), on="conversation_str", how="inner"
)

task_mapped_df = task_mapped_df.drop(columns=["conversation_str"])

print(f"Final task mapped DataFrame: {task_mapped_df.shape}")
task_mapped_df.head(2)

Final task mapped DataFrame: (9, 12)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,timezone,timestamp_local
0,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Public Relation...",Interpreters and Translators:Read written mate...,Interpreters and Translators,"Read written materials, such as legal document...",2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,Asia/Kabul,2024-11-04 05:44:03+04:30
1,"[{'role': 'user', 'content': 'You are a helpfu...","[Instructional Designers and Technologists, Te...",Instructional Designers and Technologists:Deve...,Instructional Designers and Technologists,Develop instructional materials and products f...,2024-11-09 09:44:50+00:00,Belgium,Brussels Capital,5a527b1cb1af88eb990a824042cb105bb336ee03606f4f...,Yes,Europe/Brussels,2024-11-09 10:44:50+01:00


In [ ]:
agreement = agreement_rate(
    df=task_mapped_df,
    column="selected_task_human_eval",
)
print(agreement)

## 6. Labor Transfer Analysis

In [22]:
if not LABOR_TRANSFER_LABELED_OUTPUT_PATH.exists():
    labor_transfer_labels = analyze_labor_transfer(
        client=client, df=task_mapped_df, execution_mode=execution_mode
    )
    pd.DataFrame({"label": labor_transfer_labels}).to_csv(
        LABOR_TRANSFER_LABELED_OUTPUT_PATH, index=False
    )
else:
    labor_transfer_labels = pd.read_csv(LABOR_TRANSFER_LABELED_OUTPUT_PATH)[
        "label"
    ].tolist()

print(f"Labor transfer labels: {len(labor_transfer_labels)}")
task_mapped_df["labor_transfer"] = labor_transfer_labels
print(f"Labor transfer labels assigned: {task_mapped_df.shape}")

Running 9 direct Labor Transfer calls via OpenRouter...


Labor Transfer: 100%|██████████| 9/9 [07:32<00:00, 50.25s/it]

Labor transfer labels: 9
Labor transfer labels assigned: (9, 13)


In [23]:
task_mapped_df = expand_labor_transfer_labels(
    df=task_mapped_df, label_column="labor_transfer"
)
print(f"Final DataFrame: {task_mapped_df.shape}")
task_mapped_df.head(2)

Final DataFrame: (9, 20)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,timezone,timestamp_local,interaction_type,task_match,label,lt1_reason,transferred_from,transferred_from_other,rationale,confidence
0,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Public Relation...",Interpreters and Translators:Read written mate...,Interpreters and Translators,"Read written materials, such as legal document...",2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,Asia/Kabul,2024-11-04 05:44:03+04:30,consumer,good,LT1,low_stakes,translator,NaN,User requested translation of a short figure c...,high
1,"[{'role': 'user', 'content': 'You are a helpfu...","[Instructional Designers and Technologists, Te...",Instructional Designers and Technologists:Deve...,Instructional Designers and Technologists,Develop instructional materials and products f...,2024-11-09 09:44:50+00:00,Belgium,Brussels Capital,5a527b1cb1af88eb990a824042cb105bb336ee03606f4f...,Yes,Europe/Brussels,2024-11-09 10:44:50+01:00,consumer,good,LT1,unclear_user_role,other,instructional designer,User asks ChatGPT to generate a synthetic data...,medium


In [ ]:
agreement = agreement_rate(
    df=task_mapped_df,
    column="labor_transfer_human_eval",
)
print(agreement)

# 7. Job Zones

In [25]:
job_zones_df = pd.read_excel(JOB_ZONES_PATH)
job_zones_df.head(2)

,O*NET-SOC Code,Title,Job Zone,Date,Domain Source
0,11-1011.00,Chief Executives,5,08/2023,Analyst
1,11-1011.03,Chief Sustainability Officers,5,08/2021,Analyst


In [26]:
task_mapped_df = task_mapped_df.merge(
    job_zones_df[["Title", "Job Zone"]],
    left_on="job_title",
    right_on="Title",
    how="left",
)
task_mapped_df = task_mapped_df.drop(columns=["Title"])
print(f"After merging job zones: {task_mapped_df.shape}")
task_mapped_df.head(2)

After merging job zones: (9, 21)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,...,timestamp_local,interaction_type,task_match,label,lt1_reason,transferred_from,transferred_from_other,rationale,confidence,Job Zone
0,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Public Relation...",Interpreters and Translators:Read written mate...,Interpreters and Translators,"Read written materials, such as legal document...",2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,...,2024-11-04 05:44:03+04:30,consumer,good,LT1,low_stakes,translator,NaN,User requested translation of a short figure c...,high,4.0
1,"[{'role': 'user', 'content': 'You are a helpfu...","[Instructional Designers and Technologists, Te...",Instructional Designers and Technologists:Deve...,Instructional Designers and Technologists,Develop instructional materials and products f...,2024-11-09 09:44:50+00:00,Belgium,Brussels Capital,5a527b1cb1af88eb990a824042cb105bb336ee03606f4f...,Yes,...,2024-11-09 10:44:50+01:00,consumer,good,LT1,unclear_user_role,other,instructional designer,User asks ChatGPT to generate a synthetic data...,medium,NaN


In [27]:
final_df = task_mapped_df.copy()
final_df.to_csv(FINAL_LABELED_OUTPUT_PATH, index=False)